# 4 Nov · Session 1 — What is an agent?
### Lab · Watch an agent think
**Agentic AI · PGE 5 · aivancity · Dr. Antoun Yaacoub**

This notebook is **complete**: nothing to code. You **run, read the traces and experiment**.

1. Put your key in Colab **Secrets** (🔑 icon on the left): `GROQ_API_KEY` (or `CEREBRAS_API_KEY`). Same key as in the Gen AI course.
2. Runtime → **Run all**.
3. Go back to the cells marked **🧪**: change the values in the form on the right and rerun the cell. Watch what changes.

No key, or rate-limited? Choose **Ollama (local)** in the first cell: it runs a small model on Colab's GPU (Runtime → Change runtime type → **T4 GPU**; ~3 min to download).

In [ ]:
#@title Choose the model provider { display-mode: "form" }
provider = "Groq · gpt-oss-20b (faster)" #@param ["Groq · gpt-oss-120b", "Groq · gpt-oss-20b (faster)", "Cerebras · gpt-oss-120b", "Ollama (local) · qwen2.5:7b"]
!pip install -q openai
import os, re, json, time, math, subprocess, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
from openai import OpenAI
pd.set_option("display.max_colwidth", 120)
if provider.startswith("Groq"):
    from google.colab import userdata
    BASE_URL, API_KEY = "https://api.groq.com/openai/v1", userdata.get("GROQ_API_KEY")
    MODEL = "openai/gpt-oss-20b" if "(faster)" in provider else "openai/gpt-oss-120b"
elif provider.startswith("Cerebras"):
    from google.colab import userdata
    BASE_URL, API_KEY = "https://api.cerebras.ai/v1", userdata.get("CEREBRAS_API_KEY")
    MODEL = "gpt-oss-120b"
else:
    if not shutil.which("ollama"):
        !sudo apt-get -qq install -y zstd pciutils > /dev/null
        !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL); time.sleep(5)
    !ollama pull qwen2.5:7b > /dev/null 2>&1
    BASE_URL, API_KEY = "http://localhost:11434/v1", "ollama"
    MODEL = "qwen2.5:7b"
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
def failed_generation(e):
    """Groq returns 400 'tool_use_failed' when the model writes a malformed or unexpected tool call."""
    body = getattr(e, "body", None)
    err = body.get("error", body) if isinstance(body, dict) else {}
    return err.get("failed_generation") if isinstance(err, dict) else None
def clean(r):
    """gpt-oss writes narrow no-break spaces ("10\u202fNovember"): make them normal spaces so text checks work."""
    for c in r.choices:
        if isinstance(c.message.content, str): c.message.content = c.message.content.replace("\u202f", " ").replace("\xa0", " ")
    return r
def chat_tools(**kw):
    """chat.completions.create with up to 2 retries when the model writes a malformed tool call."""
    for attempt in range(3):
        try:
            return clean(client.chat.completions.create(**kw))
        except Exception as e:
            if failed_generation(e) is None or attempt == 2: raise
            kw["temperature"] = 0.3            # small change so the retry is not identical
r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Say OK."}])
print("✅", MODEL, "answers:", r.choices[0].message.content.strip()[:40])

In [ ]:
#@title The campus world: regulations and tools { display-mode: "form" }
REGULATIONS = "# Study Regulations of the Example Institute (fictional, for teaching)\n\n## 1. Attendance\nAttendance is compulsory for all lectures, labs and project sessions. Presence is recorded at the start of each session. A student who arrives more than 15 minutes late is marked absent for that session. Three unjustified absences in the same course lead to a formal warning from the programme director.\n\n## 2. Justifying an absence\nAn absence must be justified within 48 hours through the student portal. Accepted documents are a medical certificate, a summons from a public authority, or a document from the apprenticeship employer. After 48 hours the absence is recorded as unjustified and cannot be changed, except by decision of the programme director.\n\n## 3. Apprenticeship students\nApprentices follow the same attendance rules. In addition, the school informs the employer of every absence within five working days. Time spent at school counts as working time; an unjustified absence may therefore lead to a deduction from the salary decided by the employer.\n\n## 4. Continuous assessment\nContinuous assessment counts for 50 % of the course grade. It consists of short multiple-choice quizzes taken during the course days. The lowest quiz score of each course is dropped. Quizzes missed because of a justified absence are not counted in the average.\n\n## 5. Final assessment\nThe final assessment counts for the other 50 % of the grade. It may take the form of a written exam, a project defence or a hackathon, as announced in the course syllabus. A student who misses the final assessment with a justified absence is invited to the resit session.\n\n## 6. Resit session\nThe resit session takes place in June. Only students with a course grade below 10/20 or a justified absence at the final assessment may register. The resit grade replaces the final assessment grade; continuous assessment is kept. Registration closes two weeks before the session.\n\n## 7. Use of generative AI\nStudents may use generative AI tools for coursework unless the course syllabus states otherwise. Any use must be declared in a short section of the submitted work, naming the tools and their purpose. Submitting AI-generated work as one's own without declaration is treated as plagiarism.\n\n## 8. Plagiarism and academic misconduct\nPlagiarism, cheating during an exam and falsification of documents are academic misconduct. Cases are examined by the disciplinary committee, which may cancel the grade, impose a resit, or exclude the student for up to one year.\n\n## 9. Grades and appeals\nGrades are published on the student portal within three weeks after the assessment. A student may request to see their corrected copy within ten days after publication. A written appeal to the jury must be submitted within one month of publication.\n\n## 10. Internships\nMaster students complete an internship of four to six months. The internship agreement must be signed by the company, the student and the school before the first day. An internship that starts without a signed agreement is not recognised and cannot be graded.\n\n## 11. Library\nThe library is open from 8:30 to 19:00 on weekdays and from 9:00 to 13:00 on Saturdays. Students may borrow up to five books for three weeks. Laptops can be borrowed for one day against a student card.\n\n## 12. Student representatives\nEach class elects two student representatives in October. Representatives attend the programme council twice a year and transmit questions from their classmates to the programme director.\n"
SECTIONS = [p.strip() for p in REGULATIONS.split("\n## ")[1:]]
NOW = "Thursday 6 November 2026, 14:00"            # fixed "today", so everyone gets the same results
from datetime import datetime
def now(_=""):
    """Current date and time on campus."""
    return NOW
def search_regulations(query):
    """Return the 2 sections of the study regulations that best match the query."""
    words = set(re.findall(r"[a-z]{3,}", query.lower()))
    scored = sorted(SECTIONS, key=lambda s: -len(words & set(re.findall(r"[a-z]{3,}", s.lower()))))
    return "\n---\n".join(scored[:2])
def hours_between(arg):
    """Hours between two moments, written 'YYYY-MM-DD HH:MM | YYYY-MM-DD HH:MM'."""
    a, b = [datetime.strptime(x.strip(), "%Y-%m-%d %H:%M") for x in arg.split("|")]
    return round((b - a).total_seconds() / 3600, 1)
def calculator(expression):
    """Evaluate an arithmetic expression such as '3 * (2 + 5)'."""
    if not re.fullmatch(r"[0-9+\-*/(). %]+", expression.strip()): return "Error: only numbers and + - * / ( ) are allowed"
    return eval(expression)
print(len(SECTIONS), "sections in the regulations ·", "today is", NOW)

## The agent (read it once, you do not need to change it)
A ReAct agent in plain Python: the model writes `Thought:` and `Action: tool[input]`; **our code** runs the tool and sends back `Observation: …`; repeat until `Final answer:` or `max_steps`.

In [ ]:
#@title The ReAct loop { display-mode: "form" }
ALL_TOOLS = {"now": now, "search_regulations": search_regulations, "hours_between": hours_between, "calculator": calculator}
def system_prompt(tools):
    lines = "\n".join(f"- {n}[input]: {f.__doc__}" for n, f in tools.items())
    return f"""You are a campus assistant agent. You reach the student's goal by using tools.
Available tools:
{lines}
Use exactly this format, one step per message:
Thought: what you need next
Action: tool_name[input]
Then stop and wait: you will receive "Observation: ...".
When you know the answer, write:
Thought: ...
Final answer: ...
Never write an Observation yourself. Use the tools instead of guessing rules or dates.
Write the Action as plain text in your message; do not use function calling."""

TEMPERATURE = 0
def _as_react(e):
    """Some models (gpt-oss on Groq) answer with a native tool call even when no tools are sent.
    The provider then returns an error containing the call: we turn it back into a ReAct 'Action:' line."""
    body = getattr(e, "body", None)
    err = body.get("error", body) if isinstance(body, dict) else {}
    fg = err.get("failed_generation") if isinstance(err, dict) else None
    if not fg: return None
    try:
        call = json.loads(fg); args = call.get("arguments") or {}
        arg = next(iter(args.values()), "") if isinstance(args, dict) else args
        return f"Thought: I will use {call['name']}.\nAction: {call['name']}[{arg}]"
    except Exception:
        return None
def llm(messages, stop=None):
    t0 = time.time()
    for kw in ({"stop": stop}, {}):      # some providers refuse "stop": retry without it, we cut the text ourselves
        try:
            r = clean(client.chat.completions.create(model=MODEL, messages=messages, temperature=TEMPERATURE, **kw))
            text = (r.choices[0].message.content or "").split("Observation:")[0].strip()
            u = r.usage
            return text, dict(input_tokens=u.prompt_tokens, output_tokens=u.completion_tokens, seconds=round(time.time() - t0, 2))
        except Exception as e:
            react = _as_react(e)
            if react:   # the provider sent no token count with the error: estimate it (about 4 characters per token)
                est_in = sum(len(m["content"]) for m in messages) // 4
                return react, dict(input_tokens=est_in, output_tokens=len(react) // 4, seconds=round(time.time() - t0, 2))
            if not kw: raise

def run_agent(goal, max_steps=8, tools=None, verbose=True):
    tools = ALL_TOOLS if tools is None else tools
    messages = [{"role": "system", "content": system_prompt(tools)}, {"role": "user", "content": "Goal: " + goal}]
    calls, actions = [], []
    if verbose: print("🎯 GOAL:", goal, "\n")
    for step in range(1, max_steps + 1):
        out, usage = llm(messages, stop=["Observation:"]); calls.append(usage)
        messages.append({"role": "assistant", "content": out})
        if verbose: print(f"── step {step} ({usage['input_tokens']} tokens in, {usage['seconds']} s)\n" + out.replace("Thought:", "💭 Thought:").replace("Action:", "🔧 Action:").replace("Final answer:", "✅ Final answer:"))
        if "Final answer:" in out:
            return dict(answer=out.split("Final answer:")[1].strip(), steps=step, calls=calls, actions=actions)
        m = re.search(r"Action:\s*(\w+)\[(.*)\]", out)
        if not m:
            obs = "Error: write either 'Action: tool[input]' or 'Final answer: ...'."
        elif m.group(1) not in tools:
            obs = f"Error: unknown tool '{m.group(1)}'. Available tools: {', '.join(tools)}."
        else:
            actions.append(m.group(1))
            try: obs = str(tools[m.group(1)](m.group(2)))
            except Exception as e: obs = f"Error: {e}"
        if verbose: print("👀 Observation:", obs[:400] + (" …" if len(obs) > 400 else ""), "\n")
        messages.append({"role": "user", "content": "Observation: " + obs})
    if verbose: print(f"⛔ Stopped by our code: max_steps = {max_steps} reached.")
    return dict(answer=None, steps=max_steps, calls=calls, actions=actions)
print("agent ready")

## Step 1 · The loop, live
Read every 💭 thought, 🔧 action and 👀 observation. The model only writes text; our code runs the tools.

In [ ]:
GOAL = "I missed Tuesday's lab (4 November 2026, 10:00). Can I still justify my absence?"
res = run_agent(GOAL)
print(f"\n{res['steps']} model calls · tools used: {res['actions']}")

## Step 2 · Your question
Which tools does it choose? Does it ever answer without a tool?

In [ ]:
#@title 🧪 Ask the agent
goal = "I was absent on Monday (3 Nov) with a medical certificate. Will the quiz I missed count in my average?" #@param ["I was absent on Monday (3 Nov) with a medical certificate. Will the quiz I missed count in my average?", "How many books can I borrow and for how long? If I borrow today, what is the return date?", "I arrived at 9:20 for a 9:00 lecture. Am I marked absent?", "My grade was published on 20 October 2026. Am I still allowed to see my corrected copy today?", "What is the capital of Australia?"] {allow-input: true}
res = run_agent(goal)

## Step 3 · The hard limit in code
Lower `max_steps` until the agent cannot finish. Who stops it, the model or our code?

In [ ]:
#@title 🧪 Max steps
max_steps = 2 #@param {type:"slider", min:1, max:8, step:1}
res = run_agent(GOAL, max_steps=max_steps)
print("\nanswer:", res["answer"])

## Step 4 · What a run costs
The whole history is sent again at every call. Look at the input tokens grow.

In [ ]:
res = run_agent("My grade was published on 20 October 2026. Am I still allowed to see my corrected copy today, and until when can I appeal?", verbose=False)
c = pd.DataFrame(res["calls"]); c.index = c.index + 1; c.index.name = "call"
display(c)
fig, ax = plt.subplots(figsize=(7, 3.2)); ax.bar(c.index, c.input_tokens, color="#232641", label="input tokens"); ax.bar(c.index, c.output_tokens, bottom=c.input_tokens, color="#996F11", label="output tokens")
ax.set_xlabel("model call"); ax.set_ylabel("tokens"); ax.legend(); ax.set_title("Tokens per call"); plt.show()
one = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "My grade was published on 20 October 2026. Can I still see my corrected copy?"}])
print(f"agent: {c.input_tokens.sum() + c.output_tokens.sum()} tokens in {len(c)} calls, {c.seconds.sum():.1f} s  ·  one plain call: {one.usage.total_tokens} tokens")
print("answer:", res["answer"])

## Step 5 · Take a tool away
Untick a tool. Does the agent guess, try a tool that no longer exists, or admit it cannot answer?

In [ ]:
#@title 🧪 Which tools does the agent have?
use_now = True #@param {type:"boolean"}
use_search_regulations = False #@param {type:"boolean"}
use_hours_between = True #@param {type:"boolean"}
use_calculator = True #@param {type:"boolean"}
chosen = {n: f for n, f in ALL_TOOLS.items() if globals()["use_" + n]}
print("tools:", list(chosen), "\n")
res = run_agent(GOAL, tools=chosen)

## Step 6 · Workflow or agent?
A **workflow** follows a fixed path written by us: search the regulations once, then answer in one call. The **agent** decides its own steps. Same questions, compare.

In [ ]:
def run_workflow(goal):
    t0 = time.time(); context = search_regulations(goal)
    r = client.chat.completions.create(model=MODEL, temperature=0, messages=[{"role": "user", "content":
        f"Today is {NOW}; tomorrow is Friday 2026-11-07. Answer the student using only these regulations.\n\n{context}\n\nStudent: {goal}"}])
    return dict(answer=r.choices[0].message.content.strip(), calls=1, tokens=r.usage.total_tokens, seconds=round(time.time() - t0, 1))
rows = []
for q in ["I arrived at 9:20 for a 9:00 lecture. Am I marked absent?", GOAL]:
    w = run_workflow(q); a = run_agent(q, verbose=False)
    rows.append(dict(question=q[:60], method="workflow", calls=1, tokens=w["tokens"], seconds=w["seconds"], answer=w["answer"][:150]))
    rows.append(dict(question=q[:60], method="agent", calls=len(a["calls"]), tokens=sum(x["input_tokens"] + x["output_tokens"] for x in a["calls"]),
                     seconds=round(sum(x["seconds"] for x in a["calls"]), 1), answer=(a["answer"] or "—")[:150]))
display(pd.DataFrame(rows))
print("Simple question: the workflow is enough and cheaper. Date question: does the workflow compute 4 Nov 10:00 + 48 h correctly?")

## Step 7 · Errors compound
First the theory (sliders), then the real agent run several times.

In [ ]:
#@title 🧪 p = chance each step is right, n = number of steps { run: "auto" }
p = 0.95 #@param {type:"slider", min:0.5, max:1.0, step:0.01}
n = 10 #@param {type:"slider", min:1, max:40, step:1}
ns = np.arange(1, 41)
plt.figure(figsize=(6, 3)); plt.plot(ns, p ** ns, color="#232641", lw=3); plt.axvline(n, color="#996F11", ls="--")
plt.ylim(0, 1.02); plt.xlabel("steps"); plt.ylabel("task success"); plt.title(f"{p:.2f}^{n} = {p**n:.2f}"); plt.show()

In [ ]:
#@title 🧪 Run the real agent several times (temperature 1)
runs = 5 #@param {type:"slider", min:2, max:10, step:1}
TEMPERATURE = 1.0                       # let the model vary between runs
rows = []
for k in range(runs):
    a = run_agent(GOAL, verbose=False)
    ok = bool(a["answer"]) and bool(re.search(r"too late|passed|closed|expired|no longer|exceed|beyond|past the|unjustified|cannot|can.t|not able", a["answer"], re.I))
    rows.append(dict(run=k + 1, steps=a["steps"], tools=" → ".join(a["actions"]), says_too_late="✅" if ok else "❌", answer=(a["answer"] or "—")[:110]))
t = pd.DataFrame(rows); display(t)
TEMPERATURE = 0
print(f"{(t.says_too_late == '✅').sum()}/{runs} runs reach the right conclusion (the window closed at 10:00 today).")

## Step 8 · Your project: the agent sheet
Fill the form for **your** Gen AI product turned into an agent. You will present it at the kickoff (session 6).

In [ ]:
#@title 🧪 Agent sheet
team = "Team name" #@param {type:"string"}
gen_ai_product = "RAG chatbot on our course documents" #@param {type:"string"}
agent_goal = "Answer any student question and book a study room when asked" #@param {type:"string"}
tools_read = "search_documents, get_timetable, list_free_rooms" #@param {type:"string"}
tools_write = "book_room, send_email" #@param {type:"string"}
success_check = "Correct answer with a cited section; room really booked" #@param {type:"string"}
autonomy = "3 · Agent loop" #@param ["1 · Workflow", "2 · Router / tool use", "3 · Agent loop", "4 · Multi-agent"]
max_steps_and_budget = "8 steps, 30 s, 20k tokens" #@param {type:"string"}
human_approval_before = "send_email, any booking longer than 2 h" #@param {type:"string"}
sheet = f"""# Agent sheet · {team}
| | |
|---|---|
| Gen AI product | {gen_ai_product} |
| Goal | {agent_goal} |
| Read tools | {tools_read} |
| Write tools (side effects) | {tools_write} |
| How we check success | {success_check} |
| Autonomy level | {autonomy} |
| Hard limits | {max_steps_and_budget} |
| Human approval before | {human_approval_before} |"""
open("agent_sheet.md", "w").write(sheet); display(Markdown(sheet))
print("Saved as agent_sheet.md (Files panel on the left → download).")

## Step 9 · Workflow or agent? Five real requests (ReproChem)
Five requests a chemist would send to **ReproChem**. For each one: workflow or agent, justified in one sentence.

The helper applies the five questions of the S1 slide *When not to build an agent* to our answers and flags any request where the checklist disagrees with our decision. It writes the deliverable to `agent_sheet_requests.md`.

In [ ]:
%%writefile workflow_or_agent.py
"""Workflow or agent? A decision helper for the S1 agent sheet.

The five questions come from Dr. Yaacoub's S1 deck, slide "When not to build an agent"
(Agentic AI, aivancity, 4 Nov 2026), itself based on Anthropic's "Building effective
agents" (2024). The helper does not decide for you: it applies the checklist to your
answers, compares the result with the decision you wrote, and flags disagreements.

Usage
-----
    from workflow_or_agent import Request, review, to_markdown
    rows = review(REQUESTS)
    print(to_markdown(rows))
"""
from __future__ import annotations

from dataclasses import dataclass, field

# The five questions of the lecture, in the deck's order.
QUESTIONS = {
    "steps_unknown":  "Are the steps unknown until you see the task?",
    "needs_tools":    "Does it need fresh data or actions in other systems?",
    "cheap_mistakes": "Is a wrong step cheap and easy to undo?",
    "afford_cost":    "Can you afford 10x the cost and latency of one call?",
    "auto_check":     "Can you check success automatically?",
}


@dataclass
class Request:
    """One real request a user would send, with the student's answers and decision."""
    text: str
    steps_unknown: bool
    needs_tools: bool
    cheap_mistakes: bool
    afford_cost: bool
    auto_check: bool
    decision: str                      # "workflow" or "agent"
    justification: str                 # one sentence
    pattern: str = ""                  # optional: chaining, routing, parallelisation, ...
    notes: list[str] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.decision = self.decision.strip().lower()
        if self.decision not in {"workflow", "agent"}:
            raise ValueError(f"decision must be 'workflow' or 'agent', got {self.decision!r}")


def recommend(r: Request) -> tuple[str, str, list[str]]:
    """Apply the lecture checklist. Returns (decision, autonomy level, safeguards)."""
    safeguards: list[str] = []
    if not r.cheap_mistakes:
        safeguards.append("human approval before write or costly actions")
    if not r.auto_check:
        safeguards.append("keep a human in the loop to judge success")

    if not r.steps_unknown:
        level = "1 · Workflow" if r.needs_tools or r.auto_check else "0 · Single call"
        return "workflow", level, safeguards
    if not r.afford_cost:
        safeguards.append("budget too tight for a loop: fixed workflow with a fallback to a human")
        return "workflow", "1 · Workflow", safeguards
    return "agent", "3 · Agent loop", safeguards


def review(requests: list[Request]) -> list[dict]:
    """Run the checklist on every request and flag where it disagrees with the student."""
    rows = []
    for i, r in enumerate(requests, 1):
        dec, level, safe = recommend(r)
        rows.append({
            "n": i, "request": r.text, "decision": r.decision, "justification": r.justification,
            "pattern": r.pattern, "checklist": dec, "level": level,
            "safeguards": "; ".join(safe) or "none",
            "agrees": dec == r.decision,
            "answers": {k: getattr(r, k) for k in QUESTIONS},
        })
    return rows


def to_markdown(rows: list[dict], title: str = "Workflow or agent? Five real requests") -> str:
    """Render the deliverable: one table plus the checklist answers per request."""
    yn = lambda b: "yes" if b else "no"
    out = [f"# {title}", "",
           "| # | Request | Decision | Justification (one sentence) | Level | Safeguards | Checklist agrees |",
           "|---|---|---|---|---|---|---|"]
    for r in rows:
        dec = r["decision"].capitalize() + (f" ({r['pattern']})" if r["pattern"] else "")
        out.append(f"| {r['n']} | {r['request']} | **{dec}** | {r['justification']} | {r['level']} | "
                   f"{r['safeguards']} | {'yes' if r['agrees'] else '**no, review**'} |")
    out += ["", "## Checklist answers (S1 deck, \"When not to build an agent\")", "",
            "| # | " + " | ".join(QUESTIONS.values()) + " |",
            "|---|" + "---|" * len(QUESTIONS)]
    for r in rows:
        out.append(f"| {r['n']} | " + " | ".join(yn(v) for v in r["answers"].values()) + " |")
    return "\n".join(out) + "\n"


In [ ]:
%%writefile reprochem_requests.py
"""Five real requests a chemist would send to the ReproChem agent (S1 agent sheet).

ReproChem extracts the 7 computational fields of a quantum-chemistry protocol
(functional, basis set, dispersion, solvation, charge, multiplicity, software) from
papers and internal documents, then compiles them into runnable calculations.
"""
from workflow_or_agent import Request

REQUESTS = [
    Request(
        text="Read the Computational Details of this PDF and give me the 7 fields "
             "(functional, basis set, dispersion, solvation, charge, multiplicity, software) as JSON.",
        steps_unknown=False, needs_tools=True, cheap_mistakes=True, afford_cost=True, auto_check=True,
        decision="workflow", pattern="prompt chaining",
        justification="The path never changes (find the methods section, extract into a strict schema, "
                      "flag empty fields), so code fixes the steps and the gold-annotated papers check the output."),
    Request(
        text="Turn this protocol (wB97X-D/def2-TZVP, SMD water, charge 0, singlet, Gaussian 16) "
             "into a ready-to-run input file for my geometry.",
        steps_unknown=False, needs_tools=True, cheap_mistakes=True, afford_cost=True, auto_check=True,
        decision="workflow", pattern="single compile step + validator",
        justification="Compilation is a deterministic template plus a validator (for example, no extra D3 "
                      "keyword, since wB97X-D already contains its dispersion term), so a loop would only add cost."),
    Request(
        text="For these 20 papers on conformer energies, tell me which ones report all 7 fields "
             "and which ones omit charge, multiplicity or solvation.",
        steps_unknown=False, needs_tools=True, cheap_mistakes=True, afford_cost=True, auto_check=True,
        decision="workflow", pattern="parallelisation",
        justification="It is the same extraction mapped over 20 independent papers and then aggregated, "
                      "which is the parallelisation pattern with a known number of calls."),
    Request(
        text="The paper generated conformers with CREST (GFN2-xTB) and re-optimised the lowest ones at "
             "B3LYP-D3(BJ)/def2-SVP; reproduce the ensemble for my molecule and tell me which conformer "
             "is the global minimum at their level.",
        steps_unknown=True, needs_tools=True, cheap_mistakes=False, afford_cost=True, auto_check=True,
        decision="agent", pattern="agent loop with approval",
        justification="Key details such as the energy window and how many conformers were re-optimised are "
                      "often missing, so the agent must search the SI, decide, run, check convergence and "
                      "imaginary frequencies and retry, with my approval before any cluster job is submitted."),
    Request(
        text="My relative conformer energies differ from Table 2 of the paper by up to 2 kcal/mol; find out why.",
        steps_unknown=True, needs_tools=True, cheap_mistakes=True, afford_cost=True, auto_check=True,
        decision="agent", pattern="read-only diagnosis loop",
        justification="The next check depends on the last result (D3 vs D3(BJ), solvation model, geometry level, "
                      "electronic vs Gibbs energies, temperature), and since it only reads files a wrong step is cheap."),
]


In [ ]:
#@title 🧪 Build the sheet
import importlib, workflow_or_agent, reprochem_requests
importlib.reload(workflow_or_agent); importlib.reload(reprochem_requests)
from IPython.display import Markdown, display
rows = workflow_or_agent.review(reprochem_requests.REQUESTS)
sheet = workflow_or_agent.to_markdown(rows, "ReproChem · Workflow or agent? Five real requests")
open("agent_sheet_requests.md", "w").write(sheet); display(Markdown(sheet))
bad = [r["n"] for r in rows if not r["agrees"]]
print("Checklist disagrees on request(s):", bad or "none")
print("Saved as agent_sheet_requests.md (Files panel on the left → download).")